## Example 1: Running commands with invoke.run()

In [ ]:
import invoke

# Example 1: Running commands with invoke.run()
print("Example 1: Running shell commands with invoke.run()")

# Run simple shell commands
commands = [
    'echo "Hello from invoke"',
    'pwd',
    'date',
    'whoami'
]

print("Executing shell commands:")
for cmd in commands:
    try:
        result = invoke.run(cmd, hide=True)
        print(f"  Command: {cmd}")
        print(f"    Exit code: {result.return_code}")
        print(f"    Output: {result.stdout.strip()}")
        print(f"    Success: {result.ok}")
    except Exception as e:
        print(f"  Command: {cmd} -> Error: {e}")
        raise

# Test command with error handling
print(f"\nTesting error handling:")
try:
    result = invoke.run('ls /nonexistent', warn=True, hide=True)
    print(f"  Failed command exit code: {result.return_code}")
    print(f"  Command failed: {result.failed}")
except Exception as e:
    print(f"  Error handling test: {e}")
    raise

hello = invoke.run('echo "Hello from invoke"', hide=True)
assert hello.stdout == "Hello from invoke\n" and hello.return_code == 0 and hello.ok
assert hello.command == 'echo "Hello from invoke"'
# warn=True turns a non-zero exit into a failed Result instead of an exception
assert result.failed and result.return_code != 0
# Without warn=True a non-zero exit raises UnexpectedExit carrying the Result
try:
    invoke.run('exit 3', hide=True)
    raise AssertionError("expected UnexpectedExit")
except invoke.UnexpectedExit as e:
    assert e.result.return_code == 3

## Example 2: Using Context for command execution

In [ ]:
import invoke

# Example 2: Using Context for command execution
print("Example 2: Using Context for command execution")

# Create a context
ctx = invoke.Context()
print(f"Context created: {type(ctx).__name__}")

# Run commands through context
commands = [
    'echo "Running through context"',
    'python3 --version',
    'uname -s'
]

print(f"\nExecuting commands through context:")
for cmd in commands:
    try:
        result = ctx.run(cmd, hide=True)
        print(f"  Command: {cmd}")
        print(f"    Result: {result.stdout.strip()}")
        print(f"    Command run: {result.command}")
    except Exception as e:
        print(f"  Command: {cmd} -> Error: {e}")
        raise

# Test context configuration
print(f"\nContext configuration:")
print(f"  Shell: {ctx.config.run.shell}")
print(f"  Environment vars: {len(ctx.config.run.env)} custom vars")

assert ctx.run('echo "Running through context"', hide=True).stdout == "Running through context\n"
# ctx.cd() prefixes commands with a directory change
with ctx.cd('/'):
    assert ctx.run('pwd', hide=True).stdout.strip() == '/'
# env= adds variables to the command's environment
assert ctx.run('echo $GREETING', env={'GREETING': 'hi'}, hide=True).stdout == "hi\n"

## Example 3: Creating tasks with @task decorator

In [ ]:
import invoke

# Example 3: Creating tasks with @task decorator
print("Example 3: Creating tasks with @task decorator")

# Define tasks using the @task decorator
@invoke.task
def hello_task(ctx, name="World"):
    """Say hello to someone"""
    message = f"Hello, {name}!"
    print(message)
    return message

@invoke.task  
def info_task(ctx):
    """Show system information"""
    result = ctx.run('uname -a', hide=True)
    print(f"System info: {result.stdout.strip()}")
    return result.stdout.strip()

@invoke.task
def list_task(ctx, directory="."):
    """List files in directory"""
    result = ctx.run(f'ls -la {directory}', hide=True, warn=True)
    print(f"Directory listing for {directory}:")
    print(result.stdout[:200] + "..." if len(result.stdout) > 200 else result.stdout)
    return result.ok

# Test the created tasks
print("Testing created tasks:")
ctx = invoke.Context()

print(f"  Task 1: {hello_task}")
print(f"  Task 2: {info_task}")
print(f"  Task 3: {list_task}")

# Calling a task runs its body with the given context
assert hello_task(ctx) == "Hello, World!"
assert hello_task(ctx, name="invoke") == "Hello, invoke!"
assert info_task(ctx) == ctx.run('uname -a', hide=True).stdout.strip()
assert list_task(ctx, directory="/") is True
assert hello_task.name == "hello_task"
assert isinstance(hello_task, invoke.Task)

## Example 4: Using Collection to organize tasks

In [ ]:
import invoke

# Example 4: Using Collection to organize tasks
print("Example 4: Using Collection to organize tasks")

# Create tasks
@invoke.task
def build_task(ctx):
    """Build the project"""
    print("Building project...")
    result = ctx.run('echo "Build completed"', hide=True)
    return result.ok

@invoke.task
def test_task(ctx):
    """Run tests"""
    print("Running tests...")
    result = ctx.run('echo "All tests passed"', hide=True) 
    return result.ok

@invoke.task
def deploy_task(ctx):
    """Deploy the application"""
    print("Deploying application...")
    result = ctx.run('echo "Deployment successful"', hide=True)
    return result.ok

# Create a collection and add tasks
collection = invoke.Collection()
collection.add_task(build_task)
collection.add_task(test_task)
collection.add_task(deploy_task)

print("Collection created with tasks:")
print(f"  Collection: {collection}")
print(f"  Available tasks: {list(collection.tasks.keys())}")

# Test accessing tasks from collection
print(f"\nTask details:")
for task_name, task_obj in collection.tasks.items():
    print(f"  {task_name}: {task_obj.body.__doc__.strip()}")

# Collections expose task names with dashes in place of underscores
assert sorted(collection.tasks) == ["build-task", "deploy-task", "test-task"]
assert collection["build-task"] is build_task
# An Executor runs a task by name and returns each task's result
executor = invoke.Executor(collection, config=invoke.Config())
results = executor.execute("build-task")
assert results == {build_task: True}

## Example 5: Working with Result objects and error handling

In [ ]:
import invoke

# Example 5: Working with Result objects and error handling
print("Example 5: Working with Result objects and error handling")

# Create context for testing
ctx = invoke.Context()

# Test different command results
test_commands = [
    ('echo "Success test"', "successful command"),
    ('python3 -c "print(2 + 2)"', "python calculation"),
    ('ls /tmp', "directory listing"),
    ('ls /nonexistent', "failing command")
]

print("Analyzing command results:")
for cmd, description in test_commands:
    try:
        result = ctx.run(cmd, hide=True, warn=True)
        
        print(f"  {description}:")
        print(f"    Command: {cmd}")
        print(f"    Exit code: {result.return_code}")
        print(f"    Success: {result.ok}")
        print(f"    Failed: {result.failed}")
        
        if result.stdout:
            output = result.stdout.strip()[:50]
            print(f"    Output: {output}")
            
        if result.stderr:
            error = result.stderr.strip()[:50]
            print(f"    Error: {error}")
            
    except Exception as e:
        print(f"  {description} -> Exception: {e}")
        raise

results = {description: ctx.run(cmd, hide=True, warn=True) for cmd, description in test_commands}
assert results["successful command"].stdout == "Success test\n" and results["successful command"].ok
assert results["python calculation"].stdout.strip() == "4"
assert results["failing command"].failed and results["failing command"].return_code != 0
assert results["failing command"].stderr != ""